# **Titan Tutorial**

---

<img align="right" width="300" height="300" src="../../images/hail_case_tracks.png">

This interactive tutorial takes you through the steps of how to run the Thunderstorm Identification, Tracking, Analysis and Nowcasting (Titan) application and analyze the output. Titan was originally designed as an algorithm to objectively identify and track thunderstorms from weather radar data for a weather modification experiment in South Africa in the 1980s. Now, Titan includes forecasting, storm analysis, and climatological analysis. Titan now refers to the larger system in which the original application is one component.

Titan is described in more detail in [Dixon and Wiener (1993)](https://doi.org/10.1175/1520-0426(1993)010%3C0785:TTITAA%3E2.0.CO;2).


---

## **Titan Background**

Titan identifies storm objects as a contiguous region of echo that exceeds a user-defined reflectivity threshold and minimum volume. Dual thresholds are used to deal with storm objects that briefly touch, but do not merge. Storm tracking is performed by looking for regions of overlap between storm objects at successive time intervals. Short term storm extrapolation forecasts are used to identify instances of storm merging and splitting. Titan output includes storm tracks, polygons outlining the storm objects, and storm property information (e.g., volume, area, mass, precipitation flux).

The high-level workflow for Titan is shown in the graphic below. Key steps include quality controlling the data to remove any non-meteorological or compromised echoes and gridding the data to a Cartesian grid. Once Titan is run and the tracks are produced, those data need to be converted into more user-friendly file types. 

<img align="center" width="600" src="../../images/titan_highlevel.png">

A more detailed workflow for Titan that includes each step, application, and data type is shown in the graphic below.

<img align="center" width="800" src="../../images/titan_data_flow.png">

## **Tutorial Overview**
### 1. Setup

#### Pre-processed data and prepare parameter files

Files required to run this notebook:
* Quality controlled radar data.

This radar is a 10 cm (S-band) Gematronik dual polarization radar.

The data from the morning sessions can be found here:

```
  ../data/titan/radar/raw/hail/20240806*.h5
```

### 2. Output data

After the full analysis has been run, the following derived data directories should exist:

```
  ../data/titan/titan/storms/Strathmore/20240806* (Titan binary files)
  ../data/titan/titan/ascii/Tracks2Ascii.hail.txt (Titan output converted by Tracks2Ascii)
  ../data/titan/titan/netcdf/Strathmore/titan_20240806.nc (Titan output converted by Tstorms2NetCDF)
```

### 3. Note on task cells

This notebook uses two colored cells to indicate tasks.

<div class="alert alert-block alert-info"> <b>File Task: modify parameters in text files.</b> 

These text blocks help the user modify the parameter files or other functions in *external* text files.

</div>

<div class="alert alert-block alert-warning"> <b>Cell Task: run a command in Jupyter notebook cell.</b> 

These text blocks instruct the users to run a command *in* a cell within the Jupyter notebook. If you prefer, you are welcome to copy the commands (minus the ! symbol) into a terminal window.

</div>

---

## **1. Setup**
### Environment and packages

First, we import the required python packages to run this notebook. The LROSE processing can be done with the os package and shell commands.

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")

### 1.1 Set up directories

We need to set up the required data directories. The raw radar data will be grabbed from the S3 bucket. We delete any existing files and directories specific to this tutorial to ensure we're starting with clean directories and files.

In [ ]:
#### UPDATE DIRECTORIES 

# make overall titan directory and application output directory
!mkdir -p ../data/titan

# make directory for output ascii files from Titan
!mkdir -p ../data/titan/ascii

### 1.2 Set up the environment

First, we'll set some key variables we'll need throughout the workflow.

In [ ]:
# Set directory variable to call LROSE
os.environ["LROSE_DIR"] = "/usr/local/lrose/bin"
os.environ["DATA_DIR"] = "../data/titan"

### 1.3 Get data

We will use the data that was quality controlled in the earlier part of the data during the ERAD workshop.

In [ ]:
# space for commands to find data, if needed

## **2. Set up the Titan parameter file**

This tutorial will focus on three parameters that can affect the final analysis. We have provided basic parameters including variable names, directories, etc. in order to focus on how parameter selection affects the scientific results. 

### 2.1 Reflectivity threshold

A key parameter for Titan is the minimum reflectivity value considered for storm identification: low_dbz_threshold. Storms are defined as regions with reflectivity values in excess of this value.

The default value in Titan is 35 dBZ, but this may need to change based on the air mass, storm type, etc. 

Values to consider testing: 
* 30 dBZ
* 35 dBZ
* 40 dBZ

You may wish to create different parameter files for each case to separate and save the outputs for each case. You'll just need to update the output path later.

To create a new parameter file with existing parameters, run a variation of the following command, updating the file names.

<code lang="bash">!$LROSE_DIR/Titan -params ../params/titan/Titan.params -print_params > ../params/titan/Titan.params.new</code>

### 2.2 Extension

### 2.3 Tracking variable

Titan offers two options for the storm tracking variable: 1) the 3-D reflectivity field (default) and 2) the column maximum reflectivity. This option is set in the variable use_column_max_dbz, where FALSE (default) tracks the 3-D reflectivity field and TRUE tracks the column maximum reflectivity.

Note, if use_column_max_dbz = TRUE, then the user must set the height range over which the maximum reflectivity is calulated using the following parameters: column_min_ht_km and column_max_ht_km.

## **3. Run Titan storm tracking**

Run the Titan algorithm to identify and track storms.

Titan runs on the Cartesian gridded data, using the DBZ field and optionally the VEL field to compute storm rotation.

<div class="alert alert-block alert-warning"> <b>Cell Task: Run Titan on derecho case data.</b> 
    <br>
    <br>
    Run the derecho case Titan script:
    <br>
    <br>
    <code lang="bash">!$LROSE_DIR/Titan -params ../params/titan/Titan.derecho -start "2022 05 21 12 00 00" -end "2022 05 21 20 00 00" -debug</code>
</div>

In [ ]:
# Run Titan on derecho case data
!$LROSE_DIR/Titan -params ../params/titan/Titan.derecho -start "2022 05 21 12 00 00" -end "2022 05 21 20 00 00"


## **4. Convert Titan binary output to readable format**

### 4.1 Convert Titan binary output to ASCII format for analysis.

<div class="alert alert-block alert-warning"> <b>Cell Task: Convert derecho case Titan output to ASCII.</b> 
    <br>
    <br>
    Run the derecho case ASCII conversion script:
    <br>
    <br>
    <code lang="bash">!$LROSE_DIR/Titan -params ../params/titan/Titan.derecho -start "2022 05 21 12 00 00" -end "2022 05 21 20 00 00" -debug</code>
</div>

In [ ]:
!$LROSE_DIR/Tracks2Ascii -params ../params/titan/Tracks2Ascii.derecho -f ../data/titan/titan/storms/KingCity/20220521.th5 > ../data/titan/titan/ascii/Tracks2Ascii.derecho.txt


In [ ]:
# Convert derecho case Titan output to ASCII
!$LROSE_DIR/Titan -params ../params/titan/Titan.derecho -start "2022 05 21 12 00 00" -end "2022 05 21 20 00 00"

### 4.2: Convert Titan output to NetCDF format

Convert Titan binary output to NetCDF format for further analysis.

<div class="alert alert-block alert-warning"> <b>Cell Task: Convert derecho case Titan output to NetCDF.</b> 
    <br>
    <br>
    Run the derecho case NetCDF conversion script:
    <br>
    <br>
    <code lang="bash">!$LROSE_DIR/Tstorms2NetCDF -params ../params/titan/Tstorms2NetCDF.derecho -debug -f ../data/titan/storms/KingCity/20220521.sh5</code>
</div>

In [ ]:
# Convert derecho case Titan output to NetCDF
!$LROSE_DIR/Tstorms2NetCDF -params ../params/titan/Tstorms2NetCDF.derecho -f ../data/titan/titan/storms/KingCity/20220521.sh5


## **5. Plot Output**

We'll load the necessary Python packages and plot some of the Titan output now.

In [ ]:
# Import Python packages
import pandas as pd
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import Polygon

In [ ]:
file = "../data/titan/titan/ascii/Tracks2Ascii.derecho.txt"

Open text file and adjust columns names in order to import to a pandas dataframe. 
Since the text file has irregular delimiters, we need to add some extra steps.

In [ ]:
#open file and extract column names
f = open(file)
lines = f.readlines()
f.close()

label_line_index = None  

for i, line in enumerate(lines):
    if 'labels' in line:
        label_line_index = i
        break  
labels = lines[label_line_index].split(":", 1)[1].strip().split(",")

In [ ]:
#the data lines are the ones that do not start with #
data_lines = [line.strip() for line in lines if not line.startswith("#")]

The file last three rows are labeled "parents","children","nPolySidesPolygonRays*72". Parents and children columns refer to identifiers based on merging and splitting processes. The Polygon column shows the values for the lines from the polygon centroid to each vertex, in km. There are 72 values because each line is separated 5 deg (72*5 =360). With that information and the "envelope_centroid" column, we can retrieve the cells envelopes at each timestep.

In [ ]:
rows = []
for line in data_lines:
    parts = line.split()

    try:
        # Try parsing the polygon count value (always right before 72 values)
        poly_count_index = -73  # 72 floats + 1 count (the column starts with the numnber 72, which is not part of the values)

        # Parents and children may be missing
        parent_str = parts[poly_count_index - 2]
        child_str = parts[poly_count_index - 1]

        # Handle missing values marked as "-"
        parents = int(parent_str) if parent_str != '-' else np.nan
        children = int(child_str) if child_str != '-' else np.nan

        # Polygon values: skip the count, get the next 72 values
        polygon_values = list(map(float, parts[poly_count_index + 1:]))

        # Fixed columns
        fixed_cols = parts[:poly_count_index - 2]

        # Combine into one row
        row = fixed_cols + [parents, children, polygon_values]
        rows.append(row)
    except Exception as e:
        continue

In [ ]:
# Final columns: fixed + 3 custom ones
final_labels = labels[:len(rows[0]) - 3] + ['parents', 'children', 'nPolySidesPolygonRays']

# Create DataFrame
df = pd.DataFrame(rows, columns=final_labels)

# Convert date and time columns to datetime
df['date_utc'] = pd.to_datetime(
    df['Year'].astype(str) + '-' +
    df['Month'].astype(str).str.zfill(2) + '-' +
    df['Day'].astype(str).str.zfill(2) + ' ' +
    df['Hour'].astype(str).str.zfill(2) + ':' +
    df['Min'].astype(str).str.zfill(2) + ':' +
    df['Sec'].astype(str).str.zfill(2),
    format='%Y-%m-%d %H:%M:%S',
    utc=True
)
# Print df 
print(df)

Let's explore the Titan output now!

### How does Titan work?

Titan identifies individual radar cells  within every radar volume , based on reflectivity and volume thresholds. Then, it tracks them through time using a combination and optimization scheme, and geometric logic to address splitting and merging storms. In this example, we have set the minimum reflectivity threshold to 35 dBZ. This is shown in column 'dBZThreshold'. That threshold defines the minimum reflectivity for our 'cell' entity.

### What is the Titan output?

Titan outputs cell fetures at each tracking timestep, and identifies cells within major systems, based on their interaction with neighboring cells. Therefore, each cell will have a simple and a complex identifier (“SimpleNum” and “ComplexNum”) in the Titan output text file. For example, if we are tracking a multicell system, all the individual cells tracked within the major system will have different "SimplNum" identifiers, however, they will al have the same "ComplexNum" identifier (the main multicell system).

Let's inspect our derecho case now! How many Complexes we can identify? Which one contains more tracks (e.g., single cell tracks, and split/merge processes)?

In [ ]:
# Count number of unique SimpleNum per ComplexNum
simple_counts = df.groupby('ComplexNum')['SimpleNum'].nunique().reset_index(name='NumSimple')

# Sort (optional, for better visuals)
#simple_counts = simple_counts.sort_values(by='NumSimple', ascending=False)

# Plot 1
plt.figure(figsize=(10, 8))
sns.barplot(y='ComplexNum', x='NumSimple', data=simple_counts, palette='Set3')
plt.title('Number of SimpleNum per ComplexNum', fontsize=16)
plt.xlabel('Count of Unique SimpleNum ID', fontsize=14)
plt.ylabel('ComplexNum ID', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.grid(axis='x')
plt.show()


In [ ]:
#  Filter dataframe for ComplexNum == 0 and sort by time
df0 = df[df['ComplexNum'] == "0"].copy()
df0 = df0.sort_values('date_utc')
df0['MaxDBZ(dBZ)'] = pd.to_numeric(df0['MaxDBZ(dBZ)'], errors='coerce')
df0['date_utc'] = pd.to_datetime(df0['date_utc'], errors='coerce', utc=True)

#  Plot
y_min = 30
y_max = 70
y_ticks = np.arange(y_min, y_max + 1, 5)

plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")
sns.lineplot(data=df0, x='date_utc', y='MaxDBZ(dBZ)', hue='SimpleNum', palette='gist_ncar')
plt.ylim(y_min, y_max)
plt.yticks(y_ticks,fontsize=12)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.xticks(fontsize=12)
plt.title('MaxDBZ over Time for ComplexNum = 0', fontsize=16)
plt.xlabel('Time (UTC)', fontsize=14)
plt.ylabel('MaxDBZ (dBZ)',fontsize=14)
# Remove legend
plt.legend([], [], frameon=False)
plt.tight_layout()
plt.show()


Now we can also plot the centroids of each tracked cell, in a Cartopy map, and add circles around the centroid based on how big the cell volume is in each timestep. We will also show the different cells ('SimpleNum') in different colors.

In [ ]:
df0 = df0.sort_values(['SimpleNum', 'date_utc'])
# Convert lat/lon columns to numeric, coercing errors to NaN
df0['VolCentroidLat(deg)'] = pd.to_numeric(df0['VolCentroidLat(deg)'], errors='coerce')
df0['VolCentroidLon(deg)'] = pd.to_numeric(df0['VolCentroidLon(deg)'], errors='coerce')

# Set up map plot
plt.figure(figsize=(14, 10))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-83, -80, 42, 44], crs=ccrs.PlateCarree()) 
ax.coastlines(resolution='10m')
ax.add_feature(cfeature.BORDERS, linestyle=':')
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 14}
gl.ylabel_style = {'size': 14}

# Unique SimpleNum and colors
simple_nums = df0['SimpleNum'].unique()
palette = sns.color_palette("gist_ncar", n_colors=len(simple_nums))
df0['Volume(km3)'] = pd.to_numeric(df0['Volume(km3)'], errors='coerce').fillna(0)

# We normalize the Volume for the marker sizes
for i, simple_num in enumerate(simple_nums):
    track = df0[df0['SimpleNum'] == simple_num].copy()

    lat = track['VolCentroidLat(deg)']
    lon = track['VolCentroidLon(deg)']
    vol = pd.to_numeric(track['Volume(km3)'], errors='coerce').fillna(0)

    # Scale volume: use log scale to compress range + clip to reasonable range
    sizes = np.log10(vol + 1) * 200  # +1 to avoid log(0)
    #sizes = sizes.clip(10, 300)      

    ax.plot(lon, lat, marker='o', linestyle='-', color=palette[i], alpha=0.7, transform=ccrs.PlateCarree())
    ax.scatter(lon, lat, s=sizes, color=palette[i], alpha=0.5, transform=ccrs.PlateCarree(), edgecolor='k', linewidth=0.5)

#plt.legend(title='SimpleNum', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.title('Tracks for ComplexNum = 0', fontsize= 16)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 12),
                       subplot_kw={'projection': ccrs.PlateCarree()})

# Set the map extent (lon_min, lon_max, lat_min, lat_max)
ax.set_extent([-83, -80, 42, 44], crs=ccrs.PlateCarree())
ax.coastlines()

#color by time
timesteps = df0['date_utc'].unique()
palette = sns.color_palette("gist_ncar", n_colors=len(timesteps))
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 14}
gl.ylabel_style = {'size': 14}

for idx, row in df0.iterrows():
    lat_centroid = float(row['EnvelopeCentroidLat(deg)'])
    lon_centroid = float(row['EnvelopeCentroidLon(deg)'])
    rays = row['nPolySidesPolygonRays']
    
    if not rays or len(rays) == 0:
        continue  
    
    angles = np.deg2rad(np.arange(0, 360, 5))  # 72 vertices at every 5 degrees
    rays = np.array(rays, dtype=float) #from centroid to vertex
    
    # Rays in km to degrees
    ray_x = rays * np.cos(angles)
    ray_y = rays * np.sin(angles)

    # Approximate conversion from km to degrees lat/lon
    lat_vertices = lat_centroid + ray_y / 111
    lon_vertices = lon_centroid + ray_x / (111 * np.cos(np.deg2rad(lat_centroid)))

    polygon_points = list(zip(lon_vertices, lat_vertices))
    
    poly = Polygon(polygon_points)
    time_idx = np.where(timesteps == row['date_utc'])[0][0]

    ax.add_geometries([poly], crs=ccrs.PlateCarree(),
                      edgecolor=palette[time_idx], facecolor='none', linewidth=1)
    
    ax.plot(lon_centroid, lat_centroid, marker='o',color='grey', markersize=1.5, transform=ccrs.PlateCarree())



plt.title("Track Polygons for ComplexNum=0", fontsize=16)
plt.show()
